# SuSiE fine-mapping (QTL or GWAS)

## Description

Two workflows, both backed by `pecotmr::fineMappingPipeline` (which dispatches on the S4 input class):

- **`fine_mapping`** &mdash; per-gene / per-region fine-mapping over a pre-built `QtlDataset` RDS. Fan-out goes by `--genes` (list of trait IDs, gene mode) **or** `--regions` (list of `chr:start-end` strings, region mode); exactly one must be supplied. Each task loads the same QtlDataset RDS and fine-maps one unit.
- **`fine_mapping_gwas`** &mdash; per-block fine-mapping over a list of per-block `GwasSumStats` RDS paths (typically produced by `gwas_sumstats_construct.R` / `gwas_sumstats.ipynb`). One task per RDS; each task produces one `GwasFineMappingResult` RDS. Downstream consumers (e.g. `enloc.ipynb`) concatenate the per-block outputs as needed.

## Inputs

### `fine_mapping` (QTL)

- `--qtl-dataset` &mdash; path to the QtlDataset RDS produced by `qtl_dataset.ipynb`.
- `--genes ID1 ID2 …` **or** `--regions chr:start-end …` &mdash; fan-out targets (mutually exclusive).
- `--cis-window` &mdash; bp window around each gene's TSS (gene mode only). Default 1,000,000.
- `--methods` &mdash; comma-separated fine-mapping method tokens. Default `susie`.
- `--coverage` &mdash; SuSiE credible-set coverage. Default 0.95.

### `fine_mapping_gwas` (GWAS)

- `--gwas-sumstats-list S1.rds S2.rds …` &mdash; per-block GwasSumStats RDS paths to fan out over.
- `--methods` &mdash; comma-separated fine-mapping method tokens. Default `susie`.
- `--coverage` &mdash; SuSiE credible-set coverage. Default 0.95.

### Both

- `--cwd` &mdash; output directory. Default `output`.
- `--study` &mdash; study label used in the output filename.
- `--modular-script-dir` &mdash; directory containing the per-task worker R scripts. Default `code/script`; override when SoS is invoked from a working directory that doesn't contain them.

## Outputs

- `fine_mapping`: `{cwd}/{study}.{gene|region}.finemap.rds` per fan-out unit (region strings are sanitised: `:` and `-` become `_`).
- `fine_mapping_gwas`: `{cwd}/{study}.{bn-of-gwas-sumstats-rds}.gwas_finemap.rds` per input RDS.


## Example

QTL (per-gene):
```bash
sos run pipeline/fine_mapping.ipynb fine_mapping \
    --cwd output \
    --modular-script-dir /path/to/xqtl-protocol/code/script \
    --study TEST_STUDY \
    --qtl-dataset output/TEST_STUDY.qtl_dataset.rds \
    --genes ENSG00000060237 ENSG00000234593
```

GWAS (per-block, list of GwasSumStats RDS):
```bash
sos run pipeline/fine_mapping.ipynb fine_mapping_gwas \
    --cwd output \
    --modular-script-dir /path/to/xqtl-protocol/code/script \
    --study TEST_STUDY \
    --gwas-sumstats-list output/TEST_STUDY.block1.gwas_sumstats.rds \
                         output/TEST_STUDY.block2.gwas_sumstats.rds
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: modular_script_dir = path('code/script')
# --- fine_mapping (QTL) ----------------------------------------------
parameter: qtl_dataset = path('.')
parameter: genes = []
parameter: regions = []
parameter: cis_window = 1000000
# --- fine_mapping_gwas -----------------------------------------------
parameter: gwas_sumstats_list = []
# --- shared ----------------------------------------------------------
parameter: methods = 'susie'
parameter: coverage = 0.95
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '30m'
parameter: mem = '8G'
parameter: numThreads = 1


In [ ]:
[fine_mapping]
if not qtl_dataset.is_file():
    raise ValueError("fine_mapping requires --qtl-dataset to point at an existing QtlDataset RDS.")
if bool(genes) == bool(regions):
    raise ValueError("Specify exactly one of --genes (trait IDs) or --regions (chr:start-end strings).")
fanout_items = genes if genes else regions
fanout_kind  = 'gene' if genes else 'region'
input: qtl_dataset, for_each = 'fanout_items'
output: f"{cwd}/{study}.{_fanout_items.replace(':', '_').replace('-', '_')}.finemap.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping.R \
        --qtl-dataset ${_input} \
        ${('--gene-id ' + _fanout_items + ' --cis-window ' + str(cis_window)) if fanout_kind == 'gene' else ('--region ' + _fanout_items)} \
        --methods ${methods} \
        --coverage ${coverage} \
        --output ${_output}


In [ ]:
[fine_mapping_gwas]
if not gwas_sumstats_list:
    raise ValueError("fine_mapping_gwas requires --gwas-sumstats-list pointing at one or more per-block GwasSumStats RDS files.")
gss_paths = [str(p) for p in gwas_sumstats_list]
input: gss_paths, group_by = 1
output: f"{cwd}/{study}.{_input:bnn}.gwas_finemap.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping.R \
        --gwas-sumstats ${_input} \
        --methods ${methods} \
        --coverage ${coverage} \
        --output ${_output}
